# Notebook 01: Your First DPF Simulation

**Audience:** Students at AAAPT institutions (Pakistan, Egypt, Thailand, Nigeria)  
**Assumed background:** Basic circuit theory, some plasma physics  
**Time to complete:** ~30 minutes

---

## What is a Dense Plasma Focus?

A **Dense Plasma Focus (DPF)** is one of the simplest plasma devices that can produce actual nuclear fusion reactions. It consists of two coaxial copper electrodes — a solid inner rod (the anode) surrounded by a hollow outer cylinder (the cathode) — sealed inside a chamber filled with deuterium gas at a few Torr of pressure. A capacitor bank charged to 15–30 kV discharges through the gas, creating a current sheet that accelerates plasma along the electrodes and then collapses it inward to the axis.

The physics unfolds in four phases:

```
Phase 1 — Breakdown:    Gas ionizes, a thin current sheet forms at the base of the electrodes.
Phase 2 — Rundown:      Magnetic J×B force pushes the sheet toward the anode tip (axially).
                        Think of a snowplow sweeping gas in front of it.
Phase 3 — Radial:       Sheet collapses inward (radially) toward the axis.
                        Plasma is compressed to a tiny column — this is the "focus."
Phase 4 — Pinch:        Plasma reaches maximum compression. Temperature exceeds 1 keV.
                        If the plasma is deuterium, DD fusion neutrons are produced.
```

The entire event takes **2–10 microseconds**, releases **10⁸–10¹¹ neutrons per shot**, and consumes only a few kilojoules of electrical energy. That is why DPFs are used throughout the world for neutron radiography, plasma diagnostics research, and basic fusion science.

DPFs are especially popular at universities in developing countries because they are compact (a lab bench will do), inexpensive to build, and rich in physics. The UNU-ICTP Plasma Fusion Facility — a device designed specifically for this community — has been replicated at institutions across Asia, Africa, and Latin America.

---

## The Lee Model

The **Lee model** (S. Lee, 1984–2014) is a set of ordinary differential equations that couples the external RLC circuit to the snowplow dynamics of the current sheet. It is the standard tool for DPF analysis worldwide. If you have used the Lee model Excel spreadsheet (available from the AAAPT website), you have already run a version of this model. This notebook shows you how to do the same calculation in Python, with full control over every parameter.

## Step 1: Import the simulation package

The `dpf` package is installed when you set up this repository. It contains the Lee model solver, the MHD engine, device presets, and plotting tools.

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # suppress solver convergence chatter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# The Lee model solver
from dpf.validation.lee_model_comparison import LeeModel

# Device preset library — pre-configured parameters for real DPF machines
from dpf.presets import get_preset, list_presets

print("dpf package loaded successfully.")

## Step 2: Browse available device presets

The simulator ships with presets for real devices — from the small UNU-ICTP device familiar to AAAPT students, up to the 1-megajoule PF-1000 in Warsaw. Each preset contains the circuit parameters (C, L₀, R₀, V₀) and electrode geometry.

In [ ]:
# List all available presets
all_presets = list_presets()

print(f"{'Name':<20} {'Device':<35} {'Description (truncated)'}")
print("-" * 90)
for p in all_presets:
    desc = p['description'][:45] + '...' if len(p['description']) > 45 else p['description']
    print(f"{p['name']:<20} {p['device']:<35} {desc}")

## Step 3: Load the tutorial preset and inspect the parameters

The **tutorial** preset is based on the UNU-ICTP Plasma Fusion Facility — a 3 kJ device representative of machines used in Pakistan, Thailand, Egypt, and Nigeria. Let us look at what it contains.

In [ ]:
preset = get_preset('tutorial')
cc = preset['circuit']    # circuit parameters
sp = preset['snowplow']   # snowplow (Lee model) parameters

print("=== Circuit Parameters ===")
print(f"  Capacitance C   = {cc['C']*1e6:.1f} µF")
print(f"  Charge voltage  = {cc['V0']/1e3:.0f} kV")
print(f"  Stored energy   = {0.5*cc['C']*cc['V0']**2/1e3:.1f} kJ")
print(f"  Inductance L₀   = {cc['L0']*1e9:.0f} nH")
print(f"  Resistance R₀   = {cc['R0']*1e3:.0f} mΩ")

print("\n=== Electrode Geometry ===")
print(f"  Anode radius  a = {cc['anode_radius']*100:.1f} cm")
print(f"  Cathode radius b= {cc['cathode_radius']*100:.1f} cm")
print(f"  Anode length    = {sp['anode_length']*100:.0f} cm")

print("\n=== Fill Gas ===")
# Convert rho0 to fill pressure (ideal gas, D2, T=300 K)
m_D2 = 6.687e-27   # D2 molecule mass [kg]
k_B  = 1.381e-23   # Boltzmann constant [J/K]
rho0 = preset['rho0']
p_Pa   = rho0 * k_B * 300 / m_D2
p_torr = p_Pa / 133.322
print(f"  Fill density  ρ₀ = {rho0:.4f} kg/m³")
print(f"  Fill pressure   ≈ {p_torr:.1f} Torr  ({p_Pa:.0f} Pa)")
print(f"  Fill gas          deuterium (D₂)")

## Step 4: Run the Lee model

The Lee model integrates five coupled ODEs:

- **Circuit:** `L(t)·dI/dt + R·I = V_cap`   and   `dV_cap/dt = -I/C`
- **Snowplow:** `d/dt[m(z)·vz] = F_mag - F_pressure`   (axial rundown)
- **Radial slug:** `d/dt[M·vr] = -F_rad`   (implosion)

The key coupling is that as the plasma column compresses, its inductance `L_plasma(t)` grows. This inductance back-reacts on the circuit current — which is the **current dip** you will see in the plot below.

The parameters **fc** (current fraction) and **fm** (mass fraction) are the two tuneable numbers in the Lee model. They account for the fact that not all the circuit current flows in the current sheet (some arcs through the gas or along the electrodes). Typical values: fc ≈ 0.6–0.8, fm ≈ 0.1–0.3.

In [ ]:
# Build the device parameter dictionary the Lee model expects
device_params = {
    'C':              cc['C'],
    'V0':             cc['V0'],
    'L0':             cc['L0'],
    'R0':             cc['R0'],
    'anode_radius':   cc['anode_radius'],
    'cathode_radius': cc['cathode_radius'],
    'anode_length':   sp['anode_length'],
    'fill_pressure_torr': p_torr,   # computed above
}

# Create the Lee model solver with the preset's fc and fm values
lee = LeeModel(
    current_fraction=sp['current_fraction'],   # fc = 0.70
    mass_fraction=sp['mass_fraction'],         # fm = 0.15
)

# Run the simulation — takes less than a second
result = lee.run(device_params=device_params)

print(f"Simulation complete!")
print(f"  Time steps computed : {len(result.t)}")
print(f"  Simulation span     : 0 to {result.t[-1]*1e6:.2f} µs")
print(f"  Peak current I_peak : {result.peak_current/1e3:.1f} kA")
print(f"  Time of peak        : {result.peak_current_time*1e6:.2f} µs")
print(f"  Pinch time          : {result.pinch_time*1e6:.2f} µs")
print(f"  Phases completed    : {result.phases_completed}")

## Step 5: Plot the current and voltage waveforms

These two traces are the primary diagnostic output of any DPF shot. Experimentalists measure them with a Rogowski coil (for current) and a voltage divider probe.

In [ ]:
t_us = result.t * 1e6   # convert seconds → microseconds for readability
I_kA = result.I / 1e3   # convert amperes → kiloamperes
V_kV = result.V / 1e3   # convert volts → kilovolts

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
fig.suptitle('Tutorial DPF — Lee Model Discharge Waveforms', fontsize=13, fontweight='bold')

# --- Current trace ---
ax1.plot(t_us, I_kA, 'C0', linewidth=2, label='I(t) — circuit current')
ax1.axvline(result.peak_current_time * 1e6, color='C3', linestyle='--', linewidth=1.2,
            label=f'Peak: {result.peak_current/1e3:.0f} kA at {result.peak_current_time*1e6:.2f} µs')
ax1.axvline(result.pinch_time * 1e6, color='C2', linestyle=':', linewidth=1.5,
            label=f'Pinch: {result.pinch_time*1e6:.2f} µs')
ax1.set_ylabel('Current (kA)', fontsize=11)
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Annotate the current dip — this is where the plasma is pinching
# The dip appears because growing plasma inductance resists current flow
ax1.annotate('Current dip\n(plasma pinching,\ninductance rising)',
             xy=(result.pinch_time * 1e6 * 0.92, result.peak_current * 0.72 / 1e3),
             xytext=(result.pinch_time * 1e6 * 0.60, result.peak_current * 0.50 / 1e3),
             fontsize=8, color='C3',
             arrowprops=dict(arrowstyle='->', color='C3'))

# --- Voltage trace ---
ax2.plot(t_us, V_kV, 'C1', linewidth=2, label='V_cap(t) — capacitor voltage')
ax2.axhline(0, color='k', linewidth=0.7, linestyle='--')
ax2.axvline(result.pinch_time * 1e6, color='C2', linestyle=':', linewidth=1.5,
            label='Pinch time')
ax2.set_ylabel('Capacitor Voltage (kV)', fontsize=11)
ax2.set_xlabel('Time (µs)', fontsize=11)
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nHow to read these plots:")
print("  I(t): Rises as capacitor drives current through gas. Peaks at ~T/4 of the LC period.")
print("  I(t): Then DIPS — this is the signature of the pinch. Growing L_plasma fights the current.")
print("  V(t): Falls as the capacitor discharges into the plasma.")
print("  If you see no dip → the plasma did not pinch (pressure too high, or electrodes too short).")

## Step 6: What the current waveform tells you

The current trace `I(t)` encodes the entire story of the discharge:

| Feature | What it means |
|---------|---------------|
| **Rising slope** | Energy transferring from capacitor to circuit. Slope = V/L_total. |
| **Peak current I_peak** | Maximum current delivered to the plasma. Fusion yield ~ I_peak⁴. |
| **Time of peak** | ~T/4 of LC oscillation: T = 2π√(LC). Smaller L or C → faster discharge. |
| **Current dip** | Plasma inductance L_plasma growing as pinch compresses. This is the pinch signal. |
| **Dip depth** | Deeper dip → more energy stored in the plasma column. |
| **Dip absence** | Plasma did not pinch — fill pressure too high, or rundown incomplete. |

In the lab, you will often observe the current dip on an oscilloscope and use its timing to infer when the neutron burst occurred.

In [ ]:
# Verify the LC period prediction against the simulation
import math

L_total = cc['L0']   # L0 alone (plasma adds more, but this is the unloaded estimate)
C_cap   = cc['C']
T_LC    = 2 * math.pi * math.sqrt(L_total * C_cap)
T_quarter = T_LC / 4

print(f"LC quarter-period (no plasma loading): T/4 = {T_quarter*1e6:.2f} µs")
print(f"Simulated peak current time          : {result.peak_current_time*1e6:.2f} µs")
print()
print("The simulation peak is earlier because L_plasma grows during rundown,")
print("which adds to L_total and slows the current rise partway through.")
print("Also, resistive losses reduce the peak somewhat.")

## Step 7: Track the current sheet position

The Lee model also outputs where the current sheet is in space at each moment. This helps you understand how far the plasma has traveled before the pinch.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
fig.suptitle('Tutorial DPF — Plasma Sheet Position vs Time', fontsize=13)

# Axial position during rundown phase (phase 2)
# z_sheet goes from 0 → anode_length during rundown
anode_length_cm = sp['anode_length'] * 100
z_cm = result.z_sheet * 100   # metres → centimetres

ax1.plot(t_us, z_cm, 'C4', linewidth=2)
ax1.axhline(anode_length_cm, color='gray', linestyle='--', linewidth=1,
            label=f'Anode tip ({anode_length_cm:.0f} cm)')
ax1.set_ylabel('Axial position z (cm)', fontsize=11)
ax1.set_ylim(0, anode_length_cm * 1.15)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title('Phase 2 — Axial rundown (sheet sweeps toward anode tip)')

# Radial position during pinch phase (phase 3)
# r_shock goes from cathode radius → minimum pinch radius
r_cm = result.r_shock * 100
ax2.plot(t_us, r_cm, 'C5', linewidth=2)
ax2.axhline(cc['cathode_radius'] * 100, color='gray', linestyle='--', linewidth=1,
            label=f'Cathode radius ({cc["cathode_radius"]*100:.1f} cm)')
ax2.axhline(cc['anode_radius'] * 100, color='sienna', linestyle=':', linewidth=1,
            label=f'Anode radius ({cc["anode_radius"]*100:.1f} cm)')
ax2.set_ylabel('Shock radius r (cm)', fontsize=11)
ax2.set_xlabel('Time (µs)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title('Phase 3 — Radial implosion (shock collapses toward axis)')

plt.tight_layout()
plt.show()

## Step 8: Change the fill pressure and compare

Fill pressure is the most important experimental control parameter. It determines how much gas mass the current sheet must sweep.

- **Too low:** Not enough mass → current sheet too fast → rundown before capacitor delivers full current.
- **Too high:** Too much mass → current sheet too slow → rundown incomplete, no pinch.
- **Optimal:** Balance between speed and current delivery — the "pressure optimum" for neutron yield.

Run three cases and overlay the current waveforms:

In [ ]:
# Three fill pressures to compare: low, nominal, high
pressures_torr = [1.5, 3.0, 6.0]   # Torr
colors         = ['C2', 'C0', 'C3']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Effect of Fill Pressure on Discharge Waveform', fontsize=13, fontweight='bold')

results_by_pressure = {}

for p_torr, color in zip(pressures_torr, colors):
    # Build device params with modified fill pressure
    params = dict(device_params)        # copy the base params
    params['fill_pressure_torr'] = p_torr

    res = lee.run(device_params=params)
    results_by_pressure[p_torr] = res

    t_us_p = res.t * 1e6
    label = f'{p_torr:.1f} Torr  (I_peak={res.peak_current/1e3:.0f} kA)'

    ax1.plot(t_us_p, res.I / 1e3, color=color, linewidth=2, label=label)
    ax2.plot(t_us_p, res.z_sheet * 100, color=color, linewidth=2, label=f'{p_torr:.1f} Torr')

ax1.set_xlabel('Time (µs)', fontsize=11)
ax1.set_ylabel('Current (kA)', fontsize=11)
ax1.set_title('I(t) — note dip depth and timing')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.axhline(sp['anode_length'] * 100, color='gray', linestyle='--', linewidth=1,
            label='Anode tip')
ax2.set_xlabel('Time (µs)', fontsize=11)
ax2.set_ylabel('Sheet axial position z (cm)', fontsize=11)
ax2.set_title('z(t) — sheet velocity vs fill pressure')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSummary table:")
print(f"{'Pressure (Torr)':<20} {'I_peak (kA)':<18} {'Peak time (µs)':<18} {'Pinch time (µs)'}")
print("-" * 75)
for p, res in results_by_pressure.items():
    print(f"{p:<20.1f} {res.peak_current/1e3:<18.1f} "
          f"{res.peak_current_time*1e6:<18.2f} {res.pinch_time*1e6:.2f}")

## Step 9: Interpret the results

Study the table above. You should observe:

1. **Higher pressure → later pinch.** The sheet takes longer to travel the same distance because it sweeps more mass.
2. **Higher pressure → lower peak current at the pinch.** More Ohmic losses, more inertia.
3. **Lower pressure → faster discharge**, but if pressure is too low the sheet outruns the current — the dip becomes shallow or disappears.
4. **The 3.0 Torr case** (nominal) is near optimal for this device size and stored energy.

In your laboratory, you would scan fill pressure experimentally and measure neutron yield at each point with a BF₃ detector or silver activation counter. The peak of that curve is the **pressure optimum**.

In [ ]:
# Quick sanity check: compare the quarter-period with the simulated peak time
print("Key numbers from the nominal (3.0 Torr) run:")
res_nominal = results_by_pressure[3.0]

I_peak_kA   = res_nominal.peak_current / 1e3
t_peak_us   = res_nominal.peak_current_time * 1e6
t_pinch_us  = res_nominal.pinch_time * 1e6

print(f"  Peak current   : {I_peak_kA:.1f} kA")
print(f"  Time to peak   : {t_peak_us:.2f} µs")
print(f"  Pinch time     : {t_pinch_us:.2f} µs")
print(f"  Rundown time   : {t_pinch_us - t_peak_us:.2f} µs  ← radial implosion duration")

# Implosion velocity estimate from the electrode geometry and radial implosion time
r_travel_m  = cc['cathode_radius'] - 0.1 * cc['anode_radius']   # cathode to ~r_min
t_radial_s  = (t_pinch_us - t_peak_us) * 1e-6
v_imp_km_s  = (r_travel_m / t_radial_s) / 1e3   # km/s

print(f"\nCrude implosion speed estimate:")
print(f"  Radial travel  : {r_travel_m*100:.1f} cm")
print(f"  Radial time    : {t_radial_s*1e6:.2f} µs")
print(f"  v_imp ~ {v_imp_km_s:.0f} km/s  (typical DPF: 5-15 cm/µs)")

## Exercises

Try these modifications to deepen your understanding:

1. **Change the charge voltage** from 15 kV to 20 kV. How does `I_peak` scale? (Theory predicts I_peak ∝ V₀.)

2. **Change the capacitance** from 30 µF to 15 µF at the same voltage. The stored energy halves — but the discharge is faster. Does the pinch still occur?

3. **Change `fc` from 0.70 to 0.60.** This means less current flows in the current sheet. What happens to the implosion velocity?

4. **What if `fm` (mass_fraction) is very small, say 0.05?** The sheet sweeps very little gas. How does this affect the rundown time?

---

**Next:** Notebook 02 covers the five discharge phases in detail for the PF-1000, neutron yield, and beam-target vs thermonuclear fusion.